In [0]:
%pip install --upgrade mlflow --pre databricks-agents==1.0.0rc0
dbutils.library.restartPython()

In [0]:
import mlflow
from mlflow.deployments import get_deploy_client
from mlflow.genai.scorers import global_guideline_adherence

# The guidelines below will be used to evaluate any response of the agent.
global_guidelines = ["If the request is unrelated to Databricks, the response must should be a rejection of the request",
    "If the request is related to Databricks, the response must should be concise",
    "If the request is related to Databricks and question about API, the response must have code",
    "The response must be professional."]

eval_set = [{
  "inputs": [{"role": "user", "content": "What is the difference between reduceByKey and groupByKey in Databricks Spark?"}]
}, {
  "inputs": [{"role": "user", "content": "What is the weather today?"}],
}]


# Define a very simple system-prompt agent.
@mlflow.trace(span_type="AGENT")
def llama3_agent(messages):
  SYSTEM_PROMPT = """
    You are a chatbot that answers questions about Databricks.
    For requests unrelated to Databricks, reject the request.
  """
  return get_deploy_client("databricks").predict(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    inputs={"messages": [{"role": "system", "content": SYSTEM_PROMPT}, *messages]}
  )


with mlflow.start_run(run_name="global_guidelines") as run:
  mlflow.genai.evaluate(
    data=eval_set,
    predict_fn=llama3_agent,
    scorers=[global_guideline_adherence(guidelines=global_guidelines)],
  )

In [0]:
from mlflow.genai.scorers import scorer

@scorer
def greeting(inputs, outputs, expectations=None, trace=None):
  print(outputs)
  return "hello" in outputs["choices"][0]["message"]["content"]

with mlflow.start_run(run_name="guideline_adherence") as run:
  result = mlflow.genai.evaluate(
    data=eval_set,
    predict_fn=llama3_agent,
    scorers=[greeting],
  )

In [0]:
result.metrics

In [0]:
llama3_agent(eval_set[0]["inputs"])